# TCG-pokebot — deck-AGNOSTIC training (single Save-&-Run-All)

One net that pilots the **whole field**, so self-play is honest. Built to run end-to-end on a 12h Kaggle kernel: it **checkpoints every iteration**, **stops before the kill**, **resumes across sessions**, and **persists weights** to a dataset.

## Add in the right-hand panel before Run-All
1. **Competitions** → the *Pokémon TCG AI Battle* competition (engine `cg/`, `EN_Card_Data.csv`). Must have **joined**.
2. **Datasets** (resume, optional) → your `<username>/tcg-pokebot-weights` if a previous run made one.
3. **Code** → public repo (Internet ON), or a `GITHUB_TOKEN` secret for a private repo, or set `CODE_DATASET`.
4. **Secrets** (Add-ons → Secrets) → `KAGGLE_USERNAME` + `KAGGLE_KEY` so weights persist to a dataset (else they stay in `/kaggle/working`).

This is the agnostic build; a deck-specialist net is a later, separate step.

## 1. Config

In [ ]:
USERNAME        = 'your-kaggle-username'
REPO            = 'Hakase-0/TCG-pokebot'
CODE_DATASET    = ''                               # fallback if offline: '<user>/tcg-pokebot-code'
WEIGHTS_DATASET = f'{USERNAME}/tcg-pokebot-weights'

# --- net capacity (agnostic needs more than the d=96 specialist) ---
D, N_HEADS, N_LAYERS = 192, 6, 3

# --- training ---
PHASE           = 'rollout'   # 'rollout' = strong engine-oracle teacher (slow); 'value' = fast (after sharpening)
OPP_SEARCH      = 'raw'       # opponent move: 'raw' net argmax (fast, default) | 'ismcts' symmetric search-vs-search.
                              #   Switch to 'ismcts' once you switch PHASE='value' (search becomes cheap); ~2x cost on rollout.
BC_GAMES        = 150         # agnostic BC games (combat teacher pilots the whole pool); first run only
BC_EPOCHS       = 15
ITERS           = 8           # cap raised so the TIME BUDGET governs, not the cap (~4-5 iters land per 12h session)
GAMES           = 60          # self-play games / iter (halved 120->60: ~100min/iter, so a full iter + A/B still fit)
ISMCTS_WORLDS, ISMCTS_SIMS = 2, 24
GATE_GAMES      = 80
FIELD_GAMES     = 12          # x3 internally for the agnostic field metric
TIME_BUDGET_MIN = 420         # Kaggle kills at 720; checked BETWEEN iters, so leave room for one more
                              #   in-flight iter (~100min) + cell-9 A/B (~50min) + persist before the kill

import os, sys, glob, shutil, subprocess, time
WORKDIR='/kaggle/working/TCG-pokebot'
def banner(t): print('\n'+'='*70+f'\n  {t}\n'+'='*70, flush=True)
print('config OK | net', (D,N_HEADS,N_LAYERS), '| phase', PHASE, '| opp', OPP_SEARCH)

## 2. Code

In [ ]:
banner('SETUP: code')
def have_code(p): return os.path.exists(os.path.join(p,'ismcts.py')) and os.path.exists(os.path.join(p,'selfplay_rl.py'))
def _tok():
    try:
        from kaggle_secrets import UserSecretsClient; return UserSecretsClient().get_secret('GITHUB_TOKEN')
    except Exception: return None
if not have_code(WORKDIR):
    done=False; tok=_tok()
    for url in [u for u in (f'https://{tok}@github.com/{REPO}' if tok else None, f'https://github.com/{REPO}') if u]:
        try:
            subprocess.run(['git','clone','--depth','1',url,WORKDIR],check=True); done=have_code(WORKDIR)
            if done: print('cloned from GitHub'); break
        except Exception as e: print('clone failed:', str(e)[:100])
    if not done and CODE_DATASET:
        src=f'/kaggle/input/{CODE_DATASET.split("/")[-1]}'
        cand=next((d for d,_,f in os.walk(src) if 'ismcts.py' in f),None)
        if cand:
            os.makedirs(WORKDIR,exist_ok=True)
            for f in glob.glob(os.path.join(cand,'*')):
                (shutil.copytree if os.path.isdir(f) else shutil.copy)(f,os.path.join(WORKDIR,os.path.basename(f)))
            done=have_code(WORKDIR); print('copied from code dataset:', done)
    assert done,'could not obtain code (see cell 0)'
else: print('code already present')
os.chdir(WORKDIR); sys.path.insert(0,WORKDIR)
print('cwd', os.getcwd(), '|', len(glob.glob('*.py')),'py files')


## 3. Engine + 4. Tables + 5. Deck pool (offline)

In [ ]:
banner('SETUP: engine, tables, deck pool')
# engine from the competition input
if not os.path.exists(os.path.join(WORKDIR,'cg','libcg.so')):
    so=next((os.path.join(d,'libcg.so') for d,_,fs in os.walk('/kaggle/input') if 'libcg.so' in fs),None)
    assert so,'libcg.so not found under /kaggle/input — add the competition'
    shutil.copytree(os.path.dirname(so),os.path.join(WORKDIR,'cg'),dirs_exist_ok=True); print('engine copied')
if not os.path.exists('EN_Card_Data.csv'):
    src=next((os.path.join(d,'EN_Card_Data.csv') for d,_,fs in os.walk('/kaggle/input') if 'EN_Card_Data.csv' in fs),None)
    if src: shutil.copy(src,'EN_Card_Data.csv')
os.environ['CG_LIB_PATH']=os.path.join(WORKDIR,'cg')
from cg import api; print('engine OK — cards', len(api.all_card_data()))
# tables
subprocess.run([sys.executable,'inspect_cards.py'],check=True,env={**os.environ})
print('tables:', os.path.exists('capability_table.json'))
# offline deck pool: txt -> id-csv (no internet)
from import_deck import import_decklist, write_deck_csv
for txt in sorted(glob.glob('decks/*.txt'))+sorted(glob.glob('decks/adversary/*.txt')):
    csv=txt[:-4]+'.csv'
    if not os.path.exists(csv):
        try:
            ids,_=import_decklist(open(txt).read(),'EN_Card_Data.csv')
            if len(ids)==60: write_deck_csv(ids,csv)
        except Exception as e: print('skip',os.path.basename(txt),str(e)[:60])
print('deck pool:', len(glob.glob('decks/*.csv')),'meta +', len(glob.glob('decks/adversary/*.csv')),'adversary')


In [ ]:
banner('GPU: match torch to the assigned accelerator')
# Kaggle occasionally ships a torch whose compiled archs miss the assigned GPU:
# torch.cuda.is_available() is True, but the first real kernel raises
# cudaErrorNoKernelImageForDevice (this killed the prior run mid-BC). All training
# runs in subprocesses that import torch fresh, so swapping the wheel here is picked
# up downstream. Runs ONLY when a GPU is attached but unusable; no-op otherwise.
_PROBE = ("import torch; "
          "print('NOGPU') if not torch.cuda.is_available() else "
          "[torch.zeros(1, device='cuda').add_(1.0), torch.cuda.synchronize(), "
          "print('USABLE', torch.__version__)]")
def _gpu_probe():
    r = subprocess.run([sys.executable, '-c', _PROBE], capture_output=True, text=True)
    if 'USABLE' in r.stdout: return 'USABLE', r.stdout.strip()
    if 'NOGPU'  in r.stdout: return 'NOGPU',  r.stdout.strip()
    return 'BROKEN', (r.stderr.strip().splitlines() or ['?'])[-1][:120]
tag, msg = _gpu_probe()
print(' probe:', tag, '|', msg)
if tag == 'BROKEN':
    print(' stock torch cannot run on this GPU -> reinstalling official cu121 wheel (~2-4 min)...', flush=True)
    r = subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--force-reinstall', 'torch',
                        '--index-url', 'https://download.pytorch.org/whl/cu121',
                        '--extra-index-url', 'https://pypi.org/simple'],
                       capture_output=True, text=True)
    print(' pip rc', r.returncode, '|', (r.stderr.strip() or r.stdout.strip())[-200:])
    tag, msg = _gpu_probe()
    print(' re-probe:', tag, '|', msg)
print(' -> training will use', 'CUDA' if tag == 'USABLE' else 'CPU (device() falls back automatically; no crash)')

## 6. Resume — newest checkpoint (if a weights dataset is attached)

In [ ]:
banner('RESUME')
def find_ckpt():
    roots=[f'/kaggle/input/{WEIGHTS_DATASET.split("/")[-1]}','/kaggle/working',WORKDIR]
    for pat in ('*.latest.pt','rl_model.pt','model.pt'):
        for r in roots:
            h=sorted(glob.glob(os.path.join(r,'**',pat),recursive=True),key=os.path.getmtime,reverse=True)
            if h: return h[0]
    return None
# also recover model_meta.json from a checkpoint dir so dims match on resume
WARM=find_ckpt()
if WARM:
    md_src=os.path.join(os.path.dirname(WARM),'model_meta.json')
    if os.path.exists(md_src) and not os.path.exists('model_meta.json'): shutil.copy(md_src,'model_meta.json')
NEED_BC = WARM is None
print('resume from:', WARM if WARM else 'NONE -> agnostic BC bootstrap (cell 7)')


## 7. (first run only) Agnostic BC bootstrap
Combat teacher pilots **every deck in the pool**, distilled into the bigger net. Skipped if resuming.

In [ ]:
if NEED_BC:
    banner('BC BOOTSTRAP (agnostic, combat teacher over the field)')
    subprocess.run([sys.executable,'-u','gen_selfplay_data.py','--games',str(BC_GAMES),
                    '--policy','combat','--out','data/bc.pkl','--log','logs/sp.jsonl'],check=True,env={**os.environ})
    subprocess.run([sys.executable,'-u','train_bc.py','--data','data/bc.pkl','--epochs',str(BC_EPOCHS),
                    '--out','model.pt','--d',str(D),'--n-heads',str(N_HEADS),'--n-layers',str(N_LAYERS),
                    '--log','logs/tr.jsonl'],check=True,env={**os.environ})
    WARM='model.pt'; print('BC done ->', WARM)
else: print('skipping BC; resuming from', WARM)


## 8. Agnostic RL — time-budgeted, checkpointed, harvests value data

In [ ]:
banner(f'RL (agnostic, leaf={PHASE}, opp={OPP_SEARCH})')
OUT,LATEST='rl_model.pt','rl_model.latest.pt'
cmd=[sys.executable,'-u','selfplay_rl.py','--search','ismcts','--leaf',PHASE,
     '--opp-search',OPP_SEARCH,
     '--ismcts-worlds',str(ISMCTS_WORLDS),'--ismcts-sims',str(ISMCTS_SIMS),
     '--iters',str(ITERS),'--games',str(GAMES),'--gate-games',str(GATE_GAMES),
     '--field-games',str(FIELD_GAMES),'--time-budget-min',str(TIME_BUDGET_MIN),
     '--warm',WARM,'--opp-decks','decks/','--out',OUT,'--latest-out',LATEST,
     '--dump-samples','valuedata','--log','logs/rl.jsonl']
print(' '.join(cmd),flush=True)
subprocess.run(cmd,check=True,env={**os.environ})
print('RL segment done. checkpoints:', [f for f in (OUT,LATEST) if os.path.exists(f)])

## 9. Sharpen the value head + one fast-vs-slow A/B gate
Minutes-long, frozen-body value-head fit on the harvested data; then a single A/B to decide if the **fast** value leaf can replace rollout. If yes, set `PHASE='value'` next run and the 12h limit stops binding.

In [ ]:
banner('VALUE-HEAD SHARPEN + GATE')
if glob.glob('valuedata/*.pkl') and os.path.exists('rl_model.pt'):
    subprocess.run([sys.executable,'fit_value.py','--data','valuedata','--warm','rl_model.pt',
                    '--out','rl_model.value.pt','--epochs','8','--target-blend','0.5'],check=True,env={**os.environ})
    for leaf in ('value','rollout'):
        print(f'\n[{leaf} leaf A/B, 40 games]',flush=True)
        subprocess.run([sys.executable,'arena.py','--candidate','rl_model.value.pt','--anchor','rl_model.pt',
                        '--ismcts','--leaf',leaf,'--ismcts-worlds','2','--ismcts-sims','24','--games','40',
                        '--log','logs/arena.jsonl'],env={**os.environ})
    print('\nIf value-leaf >= rollout within CI: set PHASE=value next run (fast).')
else: print('no value data yet')

## 10. Persist weights (survives the notebook)

In [ ]:
banner('PERSIST WEIGHTS')
import json
STAGE='/kaggle/working/weights'; os.makedirs(STAGE,exist_ok=True)
for f in glob.glob('rl_model*.pt')+['model.pt','model_meta.json']+glob.glob('logs/*.jsonl'):
    if os.path.exists(f): shutil.copy(f,os.path.join(STAGE,os.path.basename(f)))
print('saved to /kaggle/working/weights:', os.listdir(STAGE))
def _creds():
    try:
        from kaggle_secrets import UserSecretsClient; u=UserSecretsClient()
        os.environ['KAGGLE_USERNAME']=u.get_secret('KAGGLE_USERNAME'); os.environ['KAGGLE_KEY']=u.get_secret('KAGGLE_KEY'); return True
    except Exception as e: print('no KAGGLE_* secrets -> dataset push skipped (weights are in /kaggle/working):',str(e)[:70]); return False
if _creds():
    json.dump({'title':'tcg-pokebot-weights','id':WEIGHTS_DATASET,'licenses':[{'name':'CC0-1.0'}]},
              open(os.path.join(STAGE,'dataset-metadata.json'),'w'))
    msg=f'agnostic weights {time.strftime("%Y-%m-%d %H:%M")} phase={PHASE}'
    r=subprocess.run(['kaggle','datasets','version','-p',STAGE,'-m',msg,'--dir-mode','zip'],capture_output=True,text=True)
    if r.returncode!=0 and 'not found' in (r.stdout+r.stderr).lower():
        r=subprocess.run(['kaggle','datasets','create','-p',STAGE,'--dir-mode','zip'],capture_output=True,text=True)
    print(r.stdout[-300:] or r.stderr[-300:]); print('-> attach',WEIGHTS_DATASET,'as input next session to resume')
